# 임베딩 X 클러스터링 조합 비교 (영문)

**목적**: 3개 임베딩 모델(SBERT / E5 / SRoBERTa) X 4개 클러스터링 알고리즘(KMeans / HDBSCAN / DBSCAN / Hierarchical)의 총 12개 조합을 **`ENG_2nd_contents.csv` 전체**에 적용해 세 가지 내부 지표로 탐색하고, **Silhouette 점수가 가장 높은 조합**을 선별.

## 비교 대상

| 임베딩 | 클러스터링 |
| --- | --- |
| SBERT | KMeans / HDBSCAN / DBSCAN / Hierarchical |
| E5 | KMeans / HDBSCAN / DBSCAN / Hierarchical |
| SRoBERTa | KMeans / HDBSCAN / DBSCAN / Hierarchical |

## 내부 지표

- **Silhouette Score** (제1 기준, 높을수록 좋음, 범위 -1 ~ 1)
- **Calinski–Harabasz Index** (높을수록 좋음)
- **Davies–Bouldin Index** (낮을수록 좋음)

## K 선정

- **KMeans**: `K ∈ [2, 50]` 전부 스윕 → **Elbow(inertia) 꺾은선**(Step 6a)으로 적정 K 구간 확인 → **Silhouette 꺾은선**(Step 6b)으로 대표 K 확정(Silhouette 최대). `K=1`은 단일 군집이라 지표가 무의미해 제외.
- **Hierarchical**: K를 지정하지 않음. `distance_threshold`로 **자동 병합** → 결과 클러스터 수는 데이터에 따라 결정.
- **HDBSCAN · DBSCAN**: K 없음, 파라미터로 단발 실행.

### KMeans를 2~50까지 모두 비교할까?

| 관점 | 의견 |
| --- | --- |
| 탐색 | 2~50을 한 번에 그리면 Elbow·Silhouette 곡선 형태를 한눈에 볼 수 있어 **권장** (비교 목적에 부합). |
| K=1 | **제외** — Silhouette/CH/DB가 정의되지 않거나 항상 최악. |
| K가 너무 큼 | `K > √n` 근처부터는 과분할·노이즈 클러스터가 늘기 쉬움. 전체 약 1.9만 행이면 50까지는 실험 가능, 해석은 Elbow 꺾임 + Silhouette 피크를 함께 볼 것. |
| 비용 | 3 임베딩 × 49개 K × KMeans fit → GPU 임베딩 후 CPU에서 **수십 분~수 시간** 가능. |

## 공정성

모든 임베딩을 **L2 정규화** 한 뒤 지표 계산. Silhouette은 `metric="cosine"`, CH · DB는 정규화된 벡터의 유클리드 거리.

## 데이터

기본값 **`USE_FULL_DATA = True`** — `ENG_2nd_contents.csv` 전 행 사용. 빠른 시험만 필요하면 `USE_FULL_DATA = False` + `SAMPLE_SIZE` 설정.

## 0. 의존성 설치 (필요 시 주석 해제)

In [ ]:
# !pip install -q sentence-transformers transformers torch scikit-learn hdbscan pandas numpy matplotlib tqdm

## 1. Imports

In [ ]:
import os
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
)
from sklearn.preprocessing import normalize
import hdbscan

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "text_preprocessing":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT / "text_preprocessing").exists():
    PROJECT_ROOT = PROJECT_ROOT

OUT_DIR = PROJECT_ROOT / "out" / "text_preprocessing"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. 설정

In [ ]:
INPUT_CSV = OUT_DIR / "ENG_2nd_contents.csv"
RESULT_CSV = OUT_DIR / "clustering_comparison_results.csv"

USE_FULL_DATA = True
SAMPLE_SIZE = 5000
RANDOM_SEED = 42

K_MIN, K_MAX = 2, 50
K_RANGE = list(range(K_MIN, K_MAX + 1))  # 2, 3, ..., 50 (49 values; K=1 excluded)

HIER_DISTANCE_THRESHOLD = 0.5

HDBSCAN_MIN_CLUSTER_SIZE = 30
DBSCAN_EPS = 0.4
DBSCAN_MIN_SAMPLES = 10

if torch.cuda.is_available():
    device_str = "cuda"
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device_str = "mps"
else:
    device_str = "cpu"

print(f"project root: {PROJECT_ROOT}")
print(f"device: {device_str}")
print(f"use full data: {USE_FULL_DATA}")
print(f"KMeans sweep: K={K_MIN}..{K_MAX} ({len(K_RANGE)} values)")
print(f"hierarchical distance_threshold: {HIER_DISTANCE_THRESHOLD}")

## Step 1 — 데이터 로드

`ENG_2nd_contents.csv` 로드. `USE_FULL_DATA=True`면 전 행, `False`면 `SAMPLE_SIZE`만큼 무작위 추출.

In [ ]:
df = pd.read_csv(INPUT_CSV)
if "contents" not in df.columns and "content" in df.columns:
    df = df.rename(columns={"content": "contents"})

df = df[["contents"]].dropna()
df["contents"] = df["contents"].astype(str).str.strip()
df = df[df["contents"].str.len() > 0].reset_index(drop=True)
print("total rows after clean:", len(df))

if USE_FULL_DATA:
    work_df = df
    print("using ALL rows")
else:
    work_df = df.sample(min(SAMPLE_SIZE, len(df)), random_state=RANDOM_SEED).reset_index(drop=True)
    print(f"using SAMPLE n={len(work_df)}")

docs = work_df["contents"].tolist()
work_df.head()

## Step 2 — 임베딩 모델 카탈로그

세 모델을 base 사이즈로 맞춰 공정하게 비교. E5 계열은 문서 임베딩 시 `"passage: "` 프리픽스가 필수.

In [ ]:
EMBEDDING_CONFIGS = {
    "SBERT":    {"model": "sentence-transformers/all-mpnet-base-v2",   "prefix": ""},
    "E5":       {"model": "intfloat/e5-base-v2",                        "prefix": "passage: "},
    "SRoBERTa": {"model": "sentence-transformers/all-distilroberta-v1", "prefix": ""},
    # 메모리 여유 시 SRoBERTa를 "sentence-transformers/all-roberta-large-v1" 로 수정 가능
}
EMBEDDING_CONFIGS

## Step 3 — 임베딩 계산 (모델별 캐시)

`compare_emb_<키>.npy` 로 캐시. 캐시 행 수가 현재 샘플 수와 다르면 재계산.

In [ ]:
embeddings_by_model: dict[str, np.ndarray] = {}

for key, cfg in EMBEDDING_CONFIGS.items():
    cache_path = OUT_DIR / f"compare_emb_{key}.npy"
    if cache_path.exists():
        arr = np.load(cache_path)
        if arr.shape[0] == len(docs):
            print(f"[{key}] loaded cache {arr.shape}")
            embeddings_by_model[key] = arr
            continue
        print(f"[{key}] cache mismatch ({arr.shape[0]} != {len(docs)}), recomputing")

    print(f"[{key}] encoding with {cfg['model']}")
    model = SentenceTransformer(cfg["model"], device=device_str)
    inputs = [cfg["prefix"] + d for d in docs] if cfg["prefix"] else docs
    emb = model.encode(
        inputs,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=False,
    )
    np.save(cache_path, emb)
    embeddings_by_model[key] = emb
    print(f"[{key}] saved → {cache_path} ({emb.shape})")

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Step 4 — L2 정규화 + 지표 계산 헬퍼

- 모델 간 스케일 차이 제거를 위해 L2 정규화.
- HDBSCAN / DBSCAN 의 노이즈(-1) 은 지표 계산에서 제외.

In [ ]:
normalized_embeddings: dict[str, np.ndarray] = {
    k: normalize(v, norm="l2") for k, v in embeddings_by_model.items()
}
for k, v in normalized_embeddings.items():
    print(k, v.shape, "row-norm:", float(np.linalg.norm(v[0])))

In [ ]:
def compute_metrics(X_norm: np.ndarray, labels: np.ndarray) -> dict | None:
    labels = np.asarray(labels)
    mask = labels != -1
    uniq = np.unique(labels[mask])
    if len(uniq) < 2 or mask.sum() < len(uniq) + 1:
        return None
    Xm, Lm = X_norm[mask], labels[mask]
    return {
        "silhouette": float(silhouette_score(Xm, Lm, metric="cosine")),
        "calinski_harabasz": float(calinski_harabasz_score(Xm, Lm)),
        "davies_bouldin": float(davies_bouldin_score(Xm, Lm)),
        "n_clusters": int(len(uniq)),
        "noise_ratio": float((labels == -1).mean()),
    }

## Step 5 — 클러스터링 알고리즘 실행 함수

In [ ]:
def run_kmeans(X: np.ndarray, k: int) -> tuple[np.ndarray, float]:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_SEED)
    labels = km.fit_predict(X)
    return labels, float(km.inertia_)


def run_hier_auto(X: np.ndarray) -> np.ndarray:
    """K 미지정: distance_threshold에서 병합 중단 → 클러스터 수 자동."""
    model = AgglomerativeClustering(
        n_clusters=None,
        distance_threshold=HIER_DISTANCE_THRESHOLD,
        metric="cosine",
        linkage="average",
    )
    return model.fit_predict(X)

def run_hdbscan(X: np.ndarray) -> np.ndarray:
    model = hdbscan.HDBSCAN(min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE, metric="euclidean")
    return model.fit_predict(X)

def run_dbscan(X: np.ndarray) -> np.ndarray:
    return DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES, metric="cosine").fit_predict(X)

## Step 6 — 12개 조합 실행

- **KMeans**: `K=2..50` 스윕, `inertia` 저장 → Step 6a Elbow / Step 6b Silhouette → **Silhouette 최대 K**를 summary 대표값.
- **Hierarchical**: `distance_threshold`로 **K 자동** (스윕 없음).
- **HDBSCAN / DBSCAN**: 단발 실행.

> 전체 데이터 사용 시 Step 6(KMeans 49×3회)와 계층적 군집이 오래 걸릴 수 있습니다. 이전 `compare_emb_*.npy` 행 수가 다르면 자동 재계산됩니다.

In [ ]:
def _empty_metrics() -> dict:
    return {
        "silhouette": np.nan,
        "calinski_harabasz": np.nan,
        "davies_bouldin": np.nan,
        "n_clusters": 0,
        "noise_ratio": 0.0,
    }


def _pick_best_by_silhouette(rows: list[dict]) -> dict | None:
    best = None
    for row in rows:
        s = row.get("silhouette")
        if pd.notna(s) and (best is None or s > best["silhouette"]):
            best = row.copy()
    return best


all_runs = []
summary_rows = []

combos = []
for emb_key in EMBEDDING_CONFIGS.keys():
    for algo in ["KMeans", "Hierarchical", "HDBSCAN", "DBSCAN"]:
        combos.append((emb_key, algo))

for emb_key, algo in tqdm(combos, desc="combos"):
    X = normalized_embeddings[emb_key]

    if algo == "KMeans":
        kmeans_rows = []
        for k in tqdm(K_RANGE, desc=f"KMeans {emb_key}", leave=False):
            labels, inertia = run_kmeans(X, k)
            metrics = compute_metrics(X, labels) or _empty_metrics()
            row = {
                "embedding": emb_key,
                "algorithm": "KMeans",
                "k": k,
                "k_or_param": f"K={k}",
                "inertia": inertia,
                **metrics,
            }
            all_runs.append(row)
            kmeans_rows.append(row)
        best = _pick_best_by_silhouette(kmeans_rows)
        if best is None:
            best = {
                "embedding": emb_key,
                "algorithm": "KMeans",
                "k": np.nan,
                "k_or_param": "-",
                "inertia": np.nan,
                **_empty_metrics(),
            }
        else:
            best = best.copy()
            best["k_or_param"] = f"K={int(best['k'])} (best silhouette)"
        summary_rows.append(best)

    elif algo == "Hierarchical":
        labels = run_hier_auto(X)
        n_auto = int(len(np.unique(labels)))
        metrics = compute_metrics(X, labels) or _empty_metrics()
        row = {
            "embedding": emb_key,
            "algorithm": "Hierarchical",
            "k": n_auto,
            "k_or_param": f"auto (dist_thr={HIER_DISTANCE_THRESHOLD}, n_clusters={n_auto})",
            "inertia": np.nan,
            **metrics,
        }
        all_runs.append(row)
        summary_rows.append(row)

    else:
        labels = run_hdbscan(X) if algo == "HDBSCAN" else run_dbscan(X)
        metrics = compute_metrics(X, labels)
        param_str = (
            f"min_cluster_size={HDBSCAN_MIN_CLUSTER_SIZE}"
            if algo == "HDBSCAN"
            else f"eps={DBSCAN_EPS},min_samples={DBSCAN_MIN_SAMPLES}"
        )
        row = {
            "embedding": emb_key,
            "algorithm": algo,
            "k": np.nan,
            "k_or_param": param_str,
            **(
                metrics
                or {
                    **_empty_metrics(),
                    "n_clusters": int(len(np.unique(labels[labels != -1]))),
                    "noise_ratio": float((labels == -1).mean()),
                }
            ),
        }
        all_runs.append(row)
        summary_rows.append(row)

all_runs_df = pd.DataFrame(all_runs)
summary_df = pd.DataFrame(summary_rows)
kmeans_sweep_df = all_runs_df[all_runs_df["algorithm"] == "KMeans"].sort_values(["embedding", "k"])
print("summary rows:", len(summary_df), "| all runs:", len(all_runs_df))
kmeans_sweep_df

## Step 6a — KMeans: Elbow (Inertia vs K)

K를 늘릴수록 **inertia(WCSS)** 가 감소합니다. 꺾이는 구간(엘보) 근처가 과도한 분할 전의 적정 K 후보입니다. Step 6b Silhouette과 함께 보세요.

In [ ]:
emb_keys = list(EMBEDDING_CONFIGS.keys())
fig, axes = plt.subplots(1, len(emb_keys), figsize=(5 * len(emb_keys), 4), sharey=False)
if len(emb_keys) == 1:
    axes = [axes]

for ax, emb_key in zip(axes, emb_keys):
    sub = kmeans_sweep_df[kmeans_sweep_df["embedding"] == emb_key].sort_values("k")
    ax.plot(sub["k"], sub["inertia"], marker="o", linewidth=2, color="C0")
    ax.set_title(f"KMeans Elbow — {emb_key}")
    ax.set_xlabel("K")
    ax.set_ylabel("Inertia (WCSS)")
    ax.grid(True, alpha=0.3)

fig.suptitle("KMeans: Elbow method (look for the bend in the curve)")
plt.tight_layout()
plt.show()

## Step 6b — KMeans: Silhouette vs K

Elbow로 본 K 구간을 참고해, **Silhouette 최대 K**(검은 테두리 점)를 summary 대표값으로 사용합니다.

In [ ]:
fig, axes = plt.subplots(1, len(emb_keys), figsize=(5 * len(emb_keys), 4), sharey=True)
if len(emb_keys) == 1:
    axes = [axes]

for ax, emb_key in zip(axes, emb_keys):
    sub = kmeans_sweep_df[kmeans_sweep_df["embedding"] == emb_key].sort_values("k")
    ax.plot(sub["k"], sub["silhouette"], marker="o", linewidth=2, color="C1")
    best_row = sub.loc[sub["silhouette"].idxmax()]
    ax.scatter(
        best_row["k"],
        best_row["silhouette"],
        s=150,
        zorder=5,
        edgecolors="black",
        linewidths=1.5,
        label=f"best K={int(best_row['k'])}",
    )
    ax.set_title(f"KMeans Silhouette — {emb_key}")
    ax.set_xlabel("K")
    ax.legend(loc="best", fontsize=8)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("Silhouette (cosine)")
fig.suptitle("KMeans: Silhouette by K (★ = representative K in summary)")
plt.tight_layout()
plt.show()

## Step 7 — 결과 표 (Silhouette 제1 기준 정렬)

In [ ]:
summary_sorted = summary_df.sort_values(
    by=["silhouette", "calinski_harabasz", "davies_bouldin"],
    ascending=[False, False, True],
).reset_index(drop=True)

pd.set_option("display.float_format", lambda x: f"{x:.4f}")
pd.set_option("display.max_rows", 100)
summary_sorted

In [ ]:
# K 스윗 전체 결과 (KMeans / Hierarchical 의 모든 K 값 포함)
all_runs_df.sort_values(by="silhouette", ascending=False).reset_index(drop=True)

## Step 8 — Silhouette 막대 차트

각 조합의 Silhouette를 가로 막대로 비교. 오른쪽에 노이즈 비율/클러스터 수를 표시해 "점수는 높은데 실제 유효 클러스터는 거의 없는" 경우를 즉시 구분할 수 있음.

In [ ]:
plot_df = summary_sorted.copy()
labels = [f"{r.embedding} | {r.algorithm}" for r in plot_df.itertuples()]
scores = plot_df["silhouette"].fillna(0).values

fig, ax = plt.subplots(figsize=(9, max(4, 0.45 * len(plot_df))))
bars = ax.barh(labels[::-1], scores[::-1])
ax.set_xlabel("Silhouette Score (cosine)")
ax.set_title("Embedding x Clustering — Silhouette comparison")
ax.axvline(0, color="gray", linewidth=0.8)

for bar, (_, r) in zip(bars, list(plot_df.iloc[::-1].iterrows())):
    width = bar.get_width()
    annot = f"  n_c={r['n_clusters']}, noise={r['noise_ratio']*100:.1f}%"
    ax.text(width, bar.get_y() + bar.get_height() / 2, annot,
            va="center", ha="left", fontsize=9)

plt.tight_layout()
plt.show()

## Step 9 — 최적 조합

Silhouette 기준 1위 조합을 출력. **노이즈 비율과 클러스터 수를 함께 확인**하고 이상 없으면 [text_preprocessing_2nd_ENG.ipynb](text_preprocessing_2nd_ENG.ipynb)의 임베딩 모델 / 클러스터러를 그 조합으로 교체하면 됩니다.

In [ ]:
best = summary_sorted.iloc[0]
print("=== BEST COMBINATION (by Silhouette) ===")
print(f"embedding         : {best['embedding']}  ({EMBEDDING_CONFIGS[best['embedding']]['model']})")
print(f"algorithm         : {best['algorithm']}")
print(f"param             : {best['k_or_param']}")
print(f"n_clusters        : {best['n_clusters']}")
print(f"noise_ratio       : {best['noise_ratio']*100:.2f}%")
print(f"silhouette        : {best['silhouette']:.4f}")
print(f"calinski_harabasz : {best['calinski_harabasz']:.2f}")
print(f"davies_bouldin    : {best['davies_bouldin']:.4f}")

In [ ]:
summary_sorted.to_csv(RESULT_CSV, index=False, encoding="utf-8")
print(f"saved → {RESULT_CSV} ({len(summary_sorted)} rows)")

## 요약 및 다음 단계

- 12개 조합을 5000행 샘플로 일괄 비교하고 Silhouette/CH/DB 세 지표로 정리했습니다.
- **전체 데이터** (`USE_FULL_DATA=True`) 기준으로 비교.
- **KMeans**: K=2~50 스윕 → Step 6a Elbow → Step 6b Silhouette 최대 K를 `summary` 대표값.
- **Hierarchical**: `distance_threshold`로 K 자동 결정 (수동 K 스윕 없음).
- HDBSCAN / DBSCAN은 노이즈 비율이 높으면 점수가 높아도 실제 유효 클러스터가 적을 수 있으므로 **노이즈 비율과 클러스터 수 조건**을 함께 고려.
- DBSCAN 결과가 전부 노이즈로 나오면 설정 셀의 `DBSCAN_EPS`를 0.5–0.6으로 올려 재실행.
- SRoBERTa 품질을 더 보고 싶으면 `EMBEDDING_CONFIGS["SRoBERTa"]["model"]`을 `sentence-transformers/all-roberta-large-v1`로 교체 후 `compare_emb_SRoBERTa.npy` 삭제 후 재실행.
- 최종 결과는 `clustering_comparison_results.csv` 에서 다시 확인 가능.